In [1]:
import pandas as pd

train_df = pd.read_csv("data/train_fd001_with_rul.csv")

In [2]:
RUL_CAP = 125
train_df["RUL"] = train_df["RUL"].clip(upper=RUL_CAP)

In [3]:
drop_cols = [
    "setting_3",
    "sensor_1",
    "sensor_5",
    "sensor_6",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

In [4]:
train_df = train_df.drop(columns=drop_cols)

In [5]:
feature_cols = [col for col in train_df.columns if col not in ["engine_id", "cycle", "RUL"]]

In [6]:
print(train_df.head())
print(train_df.shape)
print(train_df.info())

   engine_id  cycle  setting_1  setting_2  sensor_2  sensor_3  sensor_4  \
0          1      1    -0.0007    -0.0004    641.82   1589.70   1400.60   
1          1      2     0.0019    -0.0003    642.15   1591.82   1403.14   
2          1      3    -0.0043     0.0003    642.35   1587.99   1404.20   
3          1      4     0.0007     0.0000    642.35   1582.79   1401.87   
4          1      5    -0.0019    -0.0002    642.37   1582.85   1406.22   

   sensor_7  sensor_8  sensor_9  sensor_11  sensor_12  sensor_13  sensor_14  \
0    554.36   2388.06   9046.19      47.47     521.66    2388.02    8138.62   
1    553.75   2388.04   9044.07      47.49     522.28    2388.07    8131.49   
2    554.26   2388.08   9052.94      47.27     522.42    2388.03    8133.23   
3    554.45   2388.11   9049.48      47.13     522.86    2388.08    8133.83   
4    554.00   2388.06   9055.15      47.28     522.19    2388.04    8133.80   

   sensor_15  sensor_17  sensor_20  sensor_21  RUL  
0     8.4195        3

In [7]:
from sklearn.preprocessing import MinMaxScaler
import joblib

scaler = MinMaxScaler()

train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])

joblib.dump(scaler, "models/scaler.pkl")

['models/scaler.pkl']

In [8]:
WINDOW_SIZE = 30

In [9]:
import numpy as np

def create_sequences(df, feature_cols, target_col="RUL", window_size=30):
    X, y = [], []

    for engine_id in df["engine_id"].unique():
        engine_data = df[df["engine_id"] == engine_id].reset_index(drop=True)

        features = engine_data[feature_cols].values
        targets = engine_data[target_col].values

        for i in range(len(engine_data) - window_size + 1):
            X.append(features[i:i + window_size])
            y.append(targets[i + window_size - 1])

    return np.array(X), np.array(y)

In [10]:
X_train, y_train = create_sequences(train_df, feature_cols, "RUL", WINDOW_SIZE)

print(X_train.shape)
print(y_train.shape)

(17731, 30, 16)
(17731,)


In [11]:
import os

os.makedirs("processed_data", exist_ok=True)

np.save("processed_data/X_train.npy", X_train)
np.save("processed_data/y_train.npy", y_train)

print("Saved successfully!")

Saved successfully!
